# Chest X-Ray Pneumonia (Transfer Learning Experiment )
- ### Analysis MetaData

## 1. Environment
분석 환경에 따라 프로젝트와 공통 라이브러리 경로를 설정합니다.

In [1]:
## PC 전용(기본 경로)
import sys

sys.path.insert(0, r"D:\DEV\AI\src")
sys.path.insert(0, r"D:\DEV\AI\projects\chest-xray-pneumonia")

In [2]:
import sys
import importlib

# ============================================================
# Environment
# ============================================================

# ENV = "remote"
ENV = "pc"
# ENV = "colab"


# ============================================================
# Project Configuration
# ============================================================

import cxp.cxp_config as cxp_config

importlib.reload(cxp_config)

config = cxp_config.configure_environment(ENV)


# ============================================================
# Python Path
# ============================================================

PROJECT_ROOT = config["PROJECT_ROOT"]
AILIB_ROOT = config["AILIB_ROOT"]

if str(AILIB_ROOT) not in sys.path:
    sys.path.insert(0, str(AILIB_ROOT))

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# ============================================================
# Project Configuration
# ============================================================

DATA_ROOT = config["DATA_ROOT"]
PROCESSED_DATA_DIR = config["PROCESSED_DATA_DIR"]

TRAIN_DIR = config["TRAIN_DIR"]
VAL_DIR = config["VAL_DIR"]
TEST_DIR = config["TEST_DIR"]

LOCAL_TRAIN_DIR = config["LOCAL_TRAIN_DIR"]
LOCAL_VAL_DIR = config["LOCAL_VAL_DIR"]
LOCAL_TEST_DIR = config["LOCAL_TEST_DIR"]

DATASET_TRAIN_DIR = config["DATASET_TRAIN_DIR"]
DATASET_VAL_DIR = config["DATASET_VAL_DIR"]
DATASET_TEST_DIR = config["DATASET_TEST_DIR"]

EXPERIMENTS_DIR = config["EXPERIMENTS_DIR"]
MODELS_DIR = config["MODELS_DIR"]
REPORTS_DIR = config["REPORTS_DIR"]

CLASS_TO_IDX = config["CLASS_TO_IDX"]
IDX_TO_CLASS = config["IDX_TO_CLASS"]


# ============================================================
# Environment Check
# ============================================================

print("=" * 60)
print("Environment Configuration")
print("=" * 60)

print(f"Environment     : {ENV}")
print(f"Project Root    : {PROJECT_ROOT}")
print(f"AI Library      : {AILIB_ROOT}")
print(f"Experiments Dir : {EXPERIMENTS_DIR}")
print(f"Reports Dir     : {REPORTS_DIR}")

print("=" * 60)

Environment Configuration
Environment     : pc
Project Root    : D:\DEV\AI\projects\chest-xray-pneumonia
AI Library      : D:\DEV\AI\src
Experiments Dir : D:\DEV\AI\projects\chest-xray-pneumonia\experiments
Reports Dir     : D:\DEV\AI\projects\chest-xray-pneumonia\reports


## 2. Experiment List

분석할 실험과 모델의 공통 설정을 정의합니다.

In [6]:
# ============================================================
# 2. Experiment List
# ============================================================

from pathlib import Path

EXPERIMENTS_DIR = Path(EXPERIMENTS_DIR)


experiment_list = sorted(
    path
    for path in EXPERIMENTS_DIR.iterdir()
    if path.is_dir()
)


print("=" * 60)
print("Available Experiments")
print("=" * 60)

for i, experiment_dir in enumerate(experiment_list, start=1):

    print(
        f"{i:02d}. "
        f"{experiment_dir.name}"
    )

print()
print(f"Total: {len(experiment_list)}")

Available Experiments
01. TL-DENSENET121-FULL
02. TL-DENSENET121-LR
03. TL-DENSENET121-PARTIAL
04. TL-RESNET50-FULL
05. TL-RESNET50-LR
06. TL-RESNET50-PARTIAL

Total: 6


## 3. Experiment Runs
 

In [7]:
# ============================================================
# 3. Experiment Runs
# ============================================================

from pathlib import Path
import pandas as pd


EXPERIMENTS_DIR = Path(EXPERIMENTS_DIR)

experiment_runs = []

for experiment_dir in sorted(EXPERIMENTS_DIR.iterdir()):

    if not experiment_dir.is_dir():
        continue

    for run_dir in sorted(experiment_dir.iterdir()):

        if not run_dir.is_dir():
            continue

        experiment_runs.append({
            "experiment": experiment_dir.name,
            "run": run_dir.name,
            "path": run_dir,
            "config": (run_dir / "config.json").exists(),
            "history": (run_dir / "history.csv").exists(),
            "model": (run_dir / "model.pth").exists(),
            "result": (run_dir / "result.json").exists(),
        })


experiment_df = pd.DataFrame(experiment_runs)

display(experiment_df)

,experiment,run,path,config,history,model,result
0,TL-DENSENET121-FULL,LR1,D:\DEV\AI\projects\chest-xray-pneumonia\experi...,True,True,True,True
1,TL-DENSENET121-FULL,LR2,D:\DEV\AI\projects\chest-xray-pneumonia\experi...,True,True,True,True
2,TL-DENSENET121-LR,LR1,D:\DEV\AI\projects\chest-xray-pneumonia\experi...,True,True,True,True
3,TL-DENSENET121-LR,LR2,D:\DEV\AI\projects\chest-xray-pneumonia\experi...,True,True,True,True
4,TL-DENSENET121-PARTIAL,LR1,D:\DEV\AI\projects\chest-xray-pneumonia\experi...,True,True,True,True
5,TL-DENSENET121-PARTIAL,LR2,D:\DEV\AI\projects\chest-xray-pneumonia\experi...,True,True,True,True
6,TL-RESNET50-FULL,LR2,D:\DEV\AI\projects\chest-xray-pneumonia\experi...,True,True,True,True
7,TL-RESNET50-FULL,LR2-EPOCH20,D:\DEV\AI\projects\chest-xray-pneumonia\experi...,True,True,True,True
8,TL-RESNET50-LR,LR1,D:\DEV\AI\projects\chest-xray-pneumonia\experi...,True,True,True,True
9,TL-RESNET50-LR,LR2,D:\DEV\AI\projects\chest-xray-pneumonia\experi...,True,True,True,True


## 4. Load Experiment Data 

In [8]:
# ============================================================
# 4. Load Experiment Data
# ============================================================

import json
from pathlib import Path

import pandas as pd


analysis_runs = []


for item in experiment_runs:

    experiment_name = item["experiment"]
    run_name = item["run"]
    run_dir = Path(item["path"])

    config_path = run_dir / "config.json"
    history_path = run_dir / "history.csv"
    result_path = run_dir / "result.json"
    model_path = run_dir / "model.pth"

    # --------------------------------------------------------
    # Required Files
    # --------------------------------------------------------

    if not config_path.exists():
        print(
            f"[Skip] Missing config: "
            f"{experiment_name} / {run_name}"
        )
        continue

    if not history_path.exists():
        print(
            f"[Skip] Missing history: "
            f"{experiment_name} / {run_name}"
        )
        continue

    if not result_path.exists():
        print(
            f"[Skip] Missing result: "
            f"{experiment_name} / {run_name}"
        )
        continue

    # --------------------------------------------------------
    # Config
    # --------------------------------------------------------

    with open(
        config_path,
        "r",
        encoding="utf-8",
    ) as f:
        config = json.load(f)

    # --------------------------------------------------------
    # History
    # --------------------------------------------------------

    history = pd.read_csv(history_path)

    # --------------------------------------------------------
    # Result
    # --------------------------------------------------------

    with open(
        result_path,
        "r",
        encoding="utf-8",
    ) as f:
        result = json.load(f)

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    analysis_runs.append({
        "experiment": experiment_name,
        "run": run_name,
        "path": run_dir,
        "config": config,
        "history": history,
        "result": result,
        "model_path": model_path,
    })


# ============================================================
# Check
# ============================================================

print("=" * 60)
print("Loaded Experiment Data")
print("=" * 60)

print(f"Total Runs: {len(analysis_runs)}")

for data in analysis_runs:

    print(
        f"{data['experiment']} / {data['run']}"
        f" | history: {len(data['history'])} epochs"
        f" | model: {data['model_path'].exists()}"
    )

Loaded Experiment Data
Total Runs: 14
TL-DENSENET121-FULL / LR1 | history: 20 epochs | model: True
TL-DENSENET121-FULL / LR2 | history: 20 epochs | model: True
TL-DENSENET121-LR / LR1 | history: 20 epochs | model: True
TL-DENSENET121-LR / LR2 | history: 20 epochs | model: True
TL-DENSENET121-PARTIAL / LR1 | history: 20 epochs | model: True
TL-DENSENET121-PARTIAL / LR2 | history: 20 epochs | model: True
TL-RESNET50-FULL / LR2 | history: 10 epochs | model: True
TL-RESNET50-FULL / LR2-EPOCH20 | history: 20 epochs | model: True
TL-RESNET50-LR / LR1 | history: 10 epochs | model: True
TL-RESNET50-LR / LR2 | history: 10 epochs | model: True
TL-RESNET50-LR / LR2-EPOCH20 | history: 20 epochs | model: True
TL-RESNET50-LR / LR3 | history: 10 epochs | model: True
TL-RESNET50-PARTIAL / LR2 | history: 10 epochs | model: True
TL-RESNET50-PARTIAL / LR2-EPOCH20 | history: 20 epochs | model: True


## 5. Experiment Metadata

In [9]:
# ============================================================
# 5. Build Experiment Metadata
# ============================================================

metadata_rows = []


for data in analysis_runs:

    config = data["config"]
    result = data["result"]

    metadata_rows.append({
        # ----------------------------------------------------
        # Experiment
        # ----------------------------------------------------
        "experiment": data["experiment"],
        "run": data["run"],

        # ----------------------------------------------------
        # Configuration
        # ----------------------------------------------------
        "model": config.get("model"),
        "image_size": config.get("image_size"),
        "channels": config.get("channels"),
        "normalization": config.get("normalization"),
        "augmentation": config.get("augmentation"),
        "batch_size": config.get("batch_size"),
        "optimizer": config.get("optimizer"),
        "learning_rate": config.get("learning_rate"),
        "weight_decay": config.get("weight_decay"),
        "num_epochs": config.get("num_epochs"),
        "use_amp": config.get("use_amp"),
        "loss": config.get("loss"),

        # ----------------------------------------------------
        # Result
        # ----------------------------------------------------
        "training_time": result.get("training_time"),
        "stopped": result.get("stopped"),
        "stop_reason": result.get("stop_reason"),
        "best_epoch": result.get("best_epoch"),
        "best_validation_loss": result.get(
            "best_validation_loss"
        ),
        "best_validation_accuracy": result.get(
            "best_validation_accuracy"
        ),
    })


metadata_df = pd.DataFrame(metadata_rows)


# ============================================================
# Check
# ============================================================

print("=" * 60)
print("Experiment Metadata")
print("=" * 60)

print("Runs   :", len(metadata_df))
print("Columns:", len(metadata_df.columns))

display(metadata_df)

Experiment Metadata
Runs   : 14
Columns: 20


,experiment,run,model,image_size,channels,normalization,augmentation,batch_size,optimizer,learning_rate,weight_decay,num_epochs,use_amp,loss,training_time,stopped,stop_reason,best_epoch,best_validation_loss,best_validation_accuracy
0,TL-DENSENET121-FULL,LR1,DenseNet,512,3,imagenet_mean_std,none,16,Adam,0.00100,0.0,20,False,CrossEntropy,5750.071838,False,None,17,0.032354,0.992337
1,TL-DENSENET121-FULL,LR2,DenseNet,512,3,imagenet_mean_std,none,16,Adam,0.00010,0.0,20,False,CrossEntropy,5783.145619,False,None,13,0.007616,0.996169
2,TL-DENSENET121-LR,LR1,DenseNet,512,3,imagenet_mean_std,none,64,Adam,0.00100,0.0,20,False,CrossEntropy,2002.678991,False,None,17,0.085301,0.969349
3,TL-DENSENET121-LR,LR2,DenseNet,512,3,imagenet_mean_std,none,64,Adam,0.00010,0.0,20,False,CrossEntropy,1998.953325,False,None,20,0.164848,0.944444
4,TL-DENSENET121-PARTIAL,LR1,DenseNet,512,3,imagenet_mean_std,none,64,Adam,0.00100,0.0,20,False,CrossEntropy,2056.535296,False,None,15,0.020797,0.995211
5,TL-DENSENET121-PARTIAL,LR2,DenseNet,512,3,imagenet_mean_std,none,64,Adam,0.00010,0.0,20,False,CrossEntropy,2857.758208,False,None,10,0.029739,0.990421
6,TL-RESNET50-FULL,LR2,ResNet,512,3,imagenet_mean_std,none,16,Adam,0.00010,0.0,10,False,CrossEntropy,2109.636307,False,None,10,0.016960,0.993295
7,TL-RESNET50-FULL,LR2-EPOCH20,ResNet,512,3,imagenet_mean_std,none,16,Adam,0.00010,0.0,20,False,CrossEntropy,4552.288362,False,None,7,0.018109,0.993295
8,TL-RESNET50-LR,LR1,ResNet,512,3,train_mean_std,none,64,Adam,0.00100,0.0,10,False,CrossEntropy,1303.692454,False,None,10,0.115350,0.954023
9,TL-RESNET50-LR,LR2,ResNet,512,3,train_mean_std,none,64,Adam,0.00010,0.0,10,False,CrossEntropy,1303.681139,False,None,10,0.239991,0.918582


In [10]:
print(metadata_df.columns.tolist())

['experiment', 'run', 'model', 'image_size', 'channels', 'normalization', 'augmentation', 'batch_size', 'optimizer', 'learning_rate', 'weight_decay', 'num_epochs', 'use_amp', 'loss', 'training_time', 'stopped', 'stop_reason', 'best_epoch', 'best_validation_loss', 'best_validation_accuracy']


## 6. Build Experiment History

In [11]:
# ============================================================
# 6. Build Experiment History
# ============================================================

history_rows = []


for data in analysis_runs:

    experiment_name = data["experiment"]
    run_name = data["run"]

    history = data["history"].copy()

    # --------------------------------------------------------
    # Experiment Information
    # --------------------------------------------------------

    history.insert(
        0,
        "experiment",
        experiment_name,
    )

    history.insert(
        1,
        "run",
        run_name,
    )

    history_rows.append(history)


experiment_history = pd.concat(
    history_rows,
    ignore_index=True,
)


# ============================================================
# Check
# ============================================================

print("=" * 60)
print("Experiment History")
print("=" * 60)

print("Rows   :", len(experiment_history))
print("Columns:", len(experiment_history.columns))

display(experiment_history.head())

Experiment History
Rows   : 230
Columns: 8


,experiment,run,epoch,train_loss,train_accuracy,validation_loss,validation_accuracy,epoch_time
0,TL-DENSENET121-FULL,LR1,1,0.180760,0.926442,2.663184,0.459770,281.572947
1,TL-DENSENET121-FULL,LR1,2,0.108294,0.960577,0.524350,0.859195,282.700387
2,TL-DENSENET121-FULL,LR1,3,0.093198,0.962740,0.105024,0.956897,288.272622
3,TL-DENSENET121-FULL,LR1,4,0.085111,0.965625,0.053893,0.979885,287.896182
4,TL-DENSENET121-FULL,LR1,5,0.069341,0.974519,0.059107,0.980843,287.753544


### 7. Save Experiment Metadata

In [ ]:
# ============================================================
# 7. Save Experiment Metadata
# ============================================================

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

metadata_path = REPORTS_DIR / "experiment_metadata.csv"
history_path = REPORTS_DIR / "experiment_history.csv"


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

metadata_df.to_csv(
    metadata_path,
    index=False,
)

experiment_history.to_csv(
    history_path,
    index=False,
)


# ============================================================
# Check
# ============================================================

print("=" * 60)
print("Saved Experiment Reports")
print("=" * 60)

print("Metadata:", metadata_path)
print("History :", history_path)

print()
print("Metadata exists:", metadata_path.exists())
print("History exists :", history_path.exists())

In [16]:
# ============================================================
# Create Experiment Summary
# ============================================================

summary_rows = []

for data in analysis_runs:

    config = data["config"]

    row = {
        "experiment": data["experiment"],
        "run": data["run"],

        # Config
        "model": config.get("model"),
        "image_size": config.get("image_size"),
        "channels": config.get("channels"),
        "normalization": config.get("normalization"),
        "augmentation": config.get("augmentation"),
        "batch_size": config.get("batch_size"),
        "optimizer": config.get("optimizer"),
        "learning_rate": config.get("learning_rate"),
        "weight_decay": config.get("weight_decay"),
        "num_epochs": config.get("num_epochs"),
        "use_amp": config.get("use_amp"),
        "loss": config.get("loss"),

        # Result
        "best_epoch": data.get("best_epoch"),
        "best_validation_loss": data.get("best_validation_loss"),
        "best_validation_accuracy": data.get("best_validation_accuracy"),
        "training_time": data.get("training_time"),
        "stopped": data.get("stopped"),
        "stop_reason": data.get("stop_reason"),
    }

    summary_rows.append(row)


experiment_summary = pd.DataFrame(summary_rows)

experiment_summary

,experiment,run,model,image_size,channels,normalization,augmentation,batch_size,optimizer,learning_rate,weight_decay,num_epochs,use_amp,loss,best_epoch,best_validation_loss,best_validation_accuracy,training_time,stopped,stop_reason
0,TL-DENSENET121-FULL,LR1,DenseNet,512,3,imagenet_mean_std,none,16,Adam,0.00100,0.0,20,False,CrossEntropy,None,None,None,None,None,None
1,TL-DENSENET121-FULL,LR2,DenseNet,512,3,imagenet_mean_std,none,16,Adam,0.00010,0.0,20,False,CrossEntropy,None,None,None,None,None,None
2,TL-DENSENET121-LR,LR1,DenseNet,512,3,imagenet_mean_std,none,64,Adam,0.00100,0.0,20,False,CrossEntropy,None,None,None,None,None,None
3,TL-DENSENET121-LR,LR2,DenseNet,512,3,imagenet_mean_std,none,64,Adam,0.00010,0.0,20,False,CrossEntropy,None,None,None,None,None,None
4,TL-DENSENET121-PARTIAL,LR1,DenseNet,512,3,imagenet_mean_std,none,64,Adam,0.00100,0.0,20,False,CrossEntropy,None,None,None,None,None,None
5,TL-DENSENET121-PARTIAL,LR2,DenseNet,512,3,imagenet_mean_std,none,64,Adam,0.00010,0.0,20,False,CrossEntropy,None,None,None,None,None,None
6,TL-RESNET50-FULL,LR2,ResNet,512,3,imagenet_mean_std,none,16,Adam,0.00010,0.0,10,False,CrossEntropy,None,None,None,None,None,None
7,TL-RESNET50-FULL,LR2-EPOCH20,ResNet,512,3,imagenet_mean_std,none,16,Adam,0.00010,0.0,20,False,CrossEntropy,None,None,None,None,None,None
8,TL-RESNET50-LR,LR1,ResNet,512,3,train_mean_std,none,64,Adam,0.00100,0.0,10,False,CrossEntropy,None,None,None,None,None,None
9,TL-RESNET50-LR,LR2,ResNet,512,3,train_mean_std,none,64,Adam,0.00010,0.0,10,False,CrossEntropy,None,None,None,None,None,None


In [18]:
# ============================================================
# Create Combined History DataFrame
# ============================================================

history_rows = []

for data in analysis_runs:

    history = data["history"].copy()

    history["experiment"] = data["experiment"]
    history["run"] = data["run"]

    history_rows.append(history)


history_df = pd.concat(
    history_rows,
    ignore_index=True
)

history_df.head()

,epoch,train_loss,train_accuracy,validation_loss,validation_accuracy,epoch_time,experiment,run
0,1,0.180760,0.926442,2.663184,0.459770,281.572947,TL-DENSENET121-FULL,LR1
1,2,0.108294,0.960577,0.524350,0.859195,282.700387,TL-DENSENET121-FULL,LR1
2,3,0.093198,0.962740,0.105024,0.956897,288.272622,TL-DENSENET121-FULL,LR1
3,4,0.085111,0.965625,0.053893,0.979885,287.896182,TL-DENSENET121-FULL,LR1
4,5,0.069341,0.974519,0.059107,0.980843,287.753544,TL-DENSENET121-FULL,LR1


In [19]:
# ============================================================
# Experiment List
# ============================================================

experiment_names = experiment_summary["experiment"].unique()

print("=" * 60)
print("Experiments")
print("=" * 60)

for i, name in enumerate(experiment_names, 1):
    runs = experiment_summary.loc[
        experiment_summary["experiment"] == name,
        "run"
    ].tolist()

    print(f"{i}. {name}")
    print(f"   Runs: {runs}")

Experiments
1. TL-DENSENET121-FULL
   Runs: ['LR1', 'LR2']
2. TL-DENSENET121-LR
   Runs: ['LR1', 'LR2']
3. TL-DENSENET121-PARTIAL
   Runs: ['LR1', 'LR2']
4. TL-RESNET50-FULL
   Runs: ['LR2', 'LR2-EPOCH20']
5. TL-RESNET50-LR
   Runs: ['LR1', 'LR2', 'LR2-EPOCH20', 'LR3']
6. TL-RESNET50-PARTIAL
   Runs: ['LR2', 'LR2-EPOCH20']


## 7. Conclusion

이번 단계에서는 각 Experiment의 Run 정보를 수집하고, `config.json`, `history.csv`, `result.json`을 기반으로 실험 Metadata와 History를 통합.

통합된 결과는 이후 실험 비교 및 분석에 사용할 수 있도록 `experiment_metadata.csv`와 `experiment_history.csv`로 저장하였다.

### Next

생성된 Experiment Metadata와 History를 기반으로 각 Experiment의 학습 결과와 Validation 성능을 분석한다.

<!-- * [03-2. ResNet50 Experiment Analysis](./03-2_chest_xray_resnet50_analysis.ipynb)
* [03-3. DenseNet121 Experiment Analysis](./03-3_chest_xray_densenet121_analysis.ipynb) -->
